# Root-Cause Classification of Whole CI Job Logs (ModernBERT, Kaggle 2x T4)

Predicts the **root cause** of a failed CI/CD job from its **whole build log** (one
label per job). This is a completely separate model from the severity classifier,
which stays an LLM (Ollama) at predict time.

### Data (real only, labels used AS-IS - no taxonomy mapping, no synthetic data)
- **TELUS Veloren subset** (built in): 2,458 whole-job logs, 13 root-cause
  categories as labeled by the TELUS regex-labeling tool (the FlaXifyer priority
  set: `misconfigured_env_variable`, `job_execution_timeout`,
  `container_registry_server_error`, `runner_pod_waiting_timeout`, `flaky_ui_test`,
  `api_gateway_deployment_error`, `external_file_invalid_format`,
  `dependency_installation_failure`, `git_transient_error`, `helm_resource_error`,
  `runner_image_pull_failure`, `host_resolution_failure`, `remote_call_timeout`).
- **FlakeStorm** (optional, gated on HF): ~4,200 logs / 30 classes. The repo is
  private; if access is granted, set `USE_FLAKESTORM=True` + `HF_TOKEN` in the
  penultimate cell and re-run. Its `category` column is used as-is.

### Tuned for Kaggle "GPU T4 x2"
- fp16 (Turing T4; not bf16), gradient checkpointing, small per-device batch
  + gradient accumulation, `accelerate.notebook_launcher` for 2-GPU data-parallel.
- Whole-job logs are windowed head+tail (20k chars -> 4,096 ModernBERT tokens);
  the same `window_text` is shipped in `ml/rootcause_model.py` for predict time.

In [ ]:
!pip install -q --force-reinstall --no-deps numpy==2.5.3 scipy==1.18.1
!pip install -q --upgrade transformers tokenizers accelerate datasets


In [ ]:
# CONFIG
import hashlib
import os
import random

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

MODEL_ID = "answerdotai/ModernBERT-base"
MAX_LEN = 4096
WINDOW_CHARS = 20000
PER_DEVICE_BATCH = 4
GRAD_ACCUM = 6
LR = 2e-5
EPOCHS = 3
DATA_CSV = None  # optional explicit path to rootcause_dataset.csv
JENKINS_CSV = None  # optional explicit path to jenkins_outputs_dataset.csv
HUB_REPO = "aziz123123v/cicd-log-rootcause"  # set to "" to skip pushing

print("root-cause training on Kaggle 2x T4 config loaded")

# stale-kernel watchdog: Colab resets pip packages per runtime, and a fresh runtime still imports
# whichever numpy it booted with. If the RUNNING numpy != the INSTALLED numpy, a restart is required.
import importlib.metadata as _md
_installed = _md.version("numpy")
if np.__version__ != _installed:
        raise RuntimeError(
            f"KERNEL STALE: running numpy {np.__version__} != installed {_installed}. "
            "Restart the runtime (top-right circular-arrow icon, or Ctrl+M then .), then run this cell again."
        )

# sanity: force a fresh, consistent import of the stack
import scipy.sparse  # noqa: F401
from sklearn.model_selection import train_test_split  # noqa: F401
import torch  # noqa: F401
from transformers import AutoTokenizer, AutoModelForSequenceClassification  # noqa: F401
import datasets  # noqa: F401
print("stack ok | numpy", np.__version__, "| scipy", scipy.__version__, "| torch", torch.__version__)

In [ ]:
# DATASET: locate the built CSV or fetch upstream and split identically to ml/build_rootcause_dataset.py
def resolve_dataset():
    candidates = [
        DATA_CSV,
        "/kaggle/input/rootcause-dataset/rootcause_dataset.csv",
        os.path.join("/kaggle/working", "rootcause_dataset.csv"),
    ]
    for c in candidates:
        if c and os.path.exists(c):
            print("using", c)
            return pd.read_csv(c, encoding="utf-8")

    import urllib.request
    import zipfile

    print("building from upstream veloren.zip ...")
    zp = "/kaggle/working/veloren.zip"
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/ahenrij/intermittent-job-failure-diagnosis/main/data/veloren.zip",
        zp,
    )
    with zipfile.ZipFile(zp) as z:
        name = next(n for n in z.namelist() if n.endswith(".csv") and "__MACOSX" not in n)
        df = pd.read_csv(z.extract(name, "/kaggle/working"), encoding="utf-8")
    df = df[["logs", "category"]].dropna(subset=["logs", "category"])
    df["logs"] = df["logs"].str.strip()
    df = df[(df["logs"] != "")].drop_duplicates(subset=["logs"]).reset_index(drop=True)
    from sklearn.model_selection import train_test_split
    idx = np.arange(len(df))
    tr, te = train_test_split(idx, test_size=0.20, random_state=SEED, stratify=df["category"])
    tr, va = train_test_split(tr, test_size=0.125, random_state=SEED, stratify=df.loc[tr, "category"])
    df["split"] = "train"
    df.loc[df.index.isin(va), "split"] = "val"
    df.loc[df.index.isin(te), "split"] = "test"
    return df


df = resolve_dataset()
print("base shape:", df.shape)

# MERGE real Jenkins log examples (adds pass_clean + per-tool classes: bandit, pylint, mypy, ruff)
JENKINS_CANDIDATES = [
    JENKINS_CSV,
    "/kaggle/input/jenkins-outputs-dataset/jenkins_outputs_dataset.csv",
    "/kaggle/working/jenkins_outputs_dataset.csv",
]
_jp = next((c for c in JENKINS_CANDIDATES if c and os.path.exists(c)), None)
if _jp:
    jdf = pd.read_csv(_jp, encoding="utf-8")
    jdf = jdf[["logs", "category", "split"]]
    jdf = jdf.dropna(subset=["logs", "category"])
    jdf["logs"] = jdf["logs"].str.strip()
    jdf = jdf[(jdf["logs"] != "")].drop_duplicates(subset=["logs"]).reset_index(drop=True)
    df = pd.concat([df, jdf], ignore_index=True).drop_duplicates(subset=["logs"]).reset_index(drop=True)
    print("merged Jenkins examples from", _jp, "| jenkins rows:", len(jdf))
else:
    print("no Jenkins CSV found - training on Veloren classes only")

print("merged shape:", df.shape)
print("columns:", list(df.columns))
print(df.groupby(["split", "category"]).size().to_string())

labels = sorted(df["category"].unique())
id2label = {i: l for i, l in enumerate(labels)}
label2id = {l: i for i, l in enumerate(labels)}
print("\nclasses (%d):" % len(labels))
print(labels)

In [ ]:
# WINDOWING (must match ml/rootcause_model.py)
def window_text(logs, chars=WINDOW_CHARS):
    logs = (logs or "").strip()
    if not logs:
        return ""
    if len(logs) <= chars:
        return logs
    half = chars // 2
    return logs[:half] + "\n[......truncated......]\n" + logs[-half:]


from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def make_split(split):
    sub = df[df["split"] == split]
    ds = Dataset.from_pandas(sub[["logs", "category"]].reset_index(drop=True))

    def tok(batch):
        texts = [window_text(t) for t in batch["logs"]]
        enc = tokenizer(texts, truncation=True, padding=False, max_length=MAX_LEN)
        return {
            "input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "label": [label2id[c] for c in batch["category"]],
        }

    return ds.map(tok, batched=True, remove_columns=["logs", "category"])


ds = {s: make_split(s) for s in ("train", "val", "test")}
for s in ds:
    print(s, len(ds[s]))

In [ ]:
# TRAINING (fp16 + grad checkpointing + 2-GPU data parallel via accelerate notebook_launcher)
import torch

from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)


def train_fn(tr_ds, va_ds):
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID, num_labels=len(labels), id2label=id2label, label2id=label2id
    )
    collator = DataCollatorWithPadding(tokenizer)
    # scale grad accumulation to the ACTUAL device count so effective batch stays ~48
    grad_accum = (GRAD_ACCUM * 2) // max(1, torch.cuda.device_count())

    def compute_metrics(ev):
        logits, y = ev
        p = np.argmax(logits, axis=-1)
        return {
            "accuracy": accuracy_score(y, p),
            "macro_f1": f1_score(y, p, average="macro", zero_division=0),
        }

    # warmup_ratio was removed in latest transformers; use explicit warmup_steps
    steps_per_epoch = max(1, len(tr_ds) // (PER_DEVICE_BATCH * max(1, torch.cuda.device_count()) * grad_accum))
    warmup_steps = max(1, int(0.1 * EPOCHS * steps_per_epoch))

    args = TrainingArguments(
        output_dir="/kaggle/working/runs",
        per_device_train_batch_size=PER_DEVICE_BATCH,
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=grad_accum,
        learning_rate=LR,
        num_train_epochs=EPOCHS,
        fp16=True,
        gradient_checkpointing=True,
        lr_scheduler_type="cosine",
        warmup_steps=warmup_steps,
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        save_total_limit=2,
        seed=SEED,
        report_to=[],
        remove_unused_columns=False,
        ddp_find_unused_parameters=False,
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tr_ds,
        eval_dataset=va_ds,
        data_collator=collator,
        compute_metrics=compute_metrics,
    )
    trainer.train()


# torch.cuda is never touched in the parent (forked children can't re-init CUDA); detect GPUs via nvidia-smi
import subprocess as _sp
_gpus = [l for l in _sp.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    capture_output=True, text=True).stdout.splitlines() if l.strip()]
print("GPUs detected:", _gpus or ["none"])

# Platform-aware launch: Kaggle has /kaggle/working and its fork-based DDP works.
# Colab-style kernels pre-hold a CUDA context, so there the 2-GPU launch hangs -> single GPU.
# Override with environment variable ROOTCAUSE_2GPU=1 (force) or 0 (force off).
_is_kaggle = os.path.exists("/kaggle/working")
_2gpu_env = os.environ.get("ROOTCAUSE_2GPU")
_allow_2gpu = (_2gpu_env == "1") if _2gpu_env is not None else (_is_kaggle and len(_gpus) >= 2)
print("platform:", "kaggle" if _is_kaggle else "other", "| 2-GPU:", _allow_2gpu)
if _allow_2gpu:
    from accelerate import notebook_launcher
    from torch.multiprocessing.spawn import ProcessRaisedException
    from torch.distributed.elastic.multiprocessing.errors import ChildFailedError
    try:
        notebook_launcher(train_fn, (ds["train"], ds["val"]), num_processes=len(_gpus))
    except (ChildFailedError, ProcessRaisedException):
        print("2-GPU launch failed on this host; retraining on ONE GPU, effective batch preserved")
        if not (os.environ.get("CUDA_VISIBLE_DEVICES") or "").strip():
            os.environ["CUDA_VISIBLE_DEVICES"] = "0"
        train_fn(ds["train"], ds["val"])
else:
    if not (os.environ.get("CUDA_VISIBLE_DEVICES") or "").strip():
        os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    print("training on ONE GPU (effective batch preserved via higher grad-accum)")
    train_fn(ds["train"], ds["val"])

In [ ]:
# TEST-SET EVALUATION (best checkpoint was saved into /kaggle/working/runs by load_best_model_at_end)
import torch

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
)

model = AutoModelForSequenceClassification.from_pretrained("/kaggle/working/runs")
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device).eval()

preds, trues = [], []
for ex in ds["test"]:
    inputs = {k: v.to(device).unsqueeze(0) for k, v in ex.items() if k in ("input_ids", "attention_mask")}
    with torch.no_grad():
        logits = model(**inputs).logits
    preds.append(int(logits.argmax(dim=-1)))
    trues.append(int(ex["label"]))

print("accuracy:", round(accuracy_score(trues, preds), 4),
      "macro_f1:", round(f1_score(trues, preds, average="macro", zero_division=0), 4))
print(classification_report(trues, preds, target_names=labels, zero_division=0))
print("confusion matrix:\n", confusion_matrix(trues, preds))

In [ ]:
# OPTIONAL: MERGE FLAKESTORM (gated on HF; requires access + HF_TOKEN). Labels used as-is.
USE_FLAKESTORM = False
if USE_FLAKESTORM:
    from datasets import load_dataset
    fs = load_dataset("ahenrij/flakestorm", split="train").to_pandas()
    fs = fs[["logs", "category"]].dropna(subset=["logs", "category"])
    fs["logs"] = fs["logs"].str.strip()
    fs = fs[(fs["logs"] != "")].drop_duplicates(subset=["logs"]).reset_index(drop=True)
    print("flakestorm rows:", len(fs), "classes:", fs["category"].nunique())
    # enable USE_FLAKESTORM above, then re-run every cell to rebuild on the union (30 classes)


In [ ]:
# PUSH TO HUGGINGFACE (HUB_REPO + HF_TOKEN). id2label is stored in model config.
if HUB_REPO:
    hf_token = os.environ.get("HF_TOKEN")
    if not hf_token:
        print("HF_TOKEN not set - model stays local at /kaggle/working/runs")
    else:
        model = AutoModelForSequenceClassification.from_pretrained("/kaggle/working/runs")
        model.push_to_hub(HUB_REPO, token=hf_token)
        tokenizer.push_to_hub(HUB_REPO, token=hf_token)
        print("pushed:", HUB_REPO)
else:
    print("HUB_REPO empty - model saved locally only")

In [ ]:
# INFERENCE DEMO on one held-out test log
import torch

if HUB_REPO:
    from transformers import AutoModelForSequenceClassification as M
    model = M.from_pretrained(HUB_REPO)
    tokenizer = AutoTokenizer.from_pretrained(HUB_REPO)
model.to(device).eval()

row = df[df["split"] == "test"].iloc[0]
text = window_text(row["logs"])
enc = tokenizer(text, truncation=True, padding=True, max_length=MAX_LEN, return_tensors="pt").to(device)
with torch.no_grad():
    probs = torch.softmax(model(**enc).logits, dim=-1)[0]
top = probs.topk(3)
print("true root cause :", row["category"])
for i, p in zip(top.indices.tolist(), top.values.tolist()):
    print(f"  pred {labels[i]:<32} {p:.3f}")